# Phase 9 — PySpark Application Architecture Experiments Notebook

This is the **worked SOLUTION notebook** for Phase 9.

Run it **top-to-bottom**. The notebook keeps all teaching code in one place so
the architecture is easy to inspect, but the design deliberately separates
responsibilities that would become modules in a real application.

The repeated workflow is:

```text
identify responsibility
    ↓
identify coupling
    ↓
state the desired contract
    ↓
refactor ONE boundary
    ↓
run the same business logic
    ↓
verify correctness
    ↓
review maintainability
```

Core question:

> **Can another engineer understand, configure, test, run, change, and troubleshoot this pipeline without reverse-engineering one giant script?**

Important:

- Examples target **PySpark 4.2.0**.
- Python examples use single-quoted strings.
- Business transformations accept DataFrames and explicit parameters.
- Paths, environments, I/O, logging, and orchestration stay outside business logic.
- The notebook does **not** perform the formal Phase 9 mastery gate, update `ROADMAP.md`, mark Phase 9 complete, or enter Phase 10.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Architecture Exercise Protocol](#architecture-exercise-protocol)
- [Experiment 1 — Decompose a Monolithic Pipeline](#experiment-1)
- [Experiment 2 — Reusable Transformation Functions](#experiment-2)
- [Experiment 3 — Parameterized Business Logic](#experiment-3)
- [Experiment 4 — Environment-Specific Configuration](#experiment-4)
- [Experiment 5 — Reader and Writer Boundaries](#experiment-5)
- [Experiment 6 — Thin Orchestration](#experiment-6)
- [Experiment 7 — Useful Logging](#experiment-7)
- [Experiment 8 — Exception Handling](#experiment-8)
- [Experiment 9 — Deterministic Behavior](#experiment-9)
- [Experiment 10 — Testable Transformation Design](#experiment-10)
- [Experiment 11 — Dependency Direction and Coupling](#experiment-11)
- [Experiment 12 — Dependency Management and Packaging](#experiment-12)
- [Applied Phase 9 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
orders_df
= one row per order_id

customers_df
= one row per customer_id

customer_history_df
= one row per customer_record_id
```

The applied pipeline deliberately changes grain:

```text
order grain
    ↓
completed/qualified order grain
    ↓
enriched order grain
    ↓
province grain
```

[Back to Table of Contents](#toc)


In [ ]:
from dataclasses import dataclass
from datetime import date
from decimal import Decimal
import logging
from pathlib import Path
from tempfile import TemporaryDirectory

from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.types import DecimalType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType


spark = (
    SparkSession.builder
    .appName('phase_09_application_architecture_experiments')
    .master('local[4]')
    # Keep the teaching workload small and predictable.
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
ORDERS_SCHEMA = StructType(
    [
        StructField('order_id', StringType(), nullable=False),
        StructField('customer_id', StringType(), nullable=False),
        StructField('order_date', DateType(), nullable=False),
        StructField('order_status', StringType(), nullable=False),
        StructField('net_sales', DecimalType(12, 2), nullable=False),
    ]
)

CUSTOMERS_SCHEMA = StructType(
    [
        StructField('customer_id', StringType(), nullable=False),
        StructField('customer_name', StringType(), nullable=False),
        StructField('province', StringType(), nullable=False),
        StructField('customer_segment', StringType(), nullable=False),
    ]
)

CUSTOMER_HISTORY_SCHEMA = StructType(
    [
        StructField('customer_record_id', StringType(), nullable=False),
        StructField('customer_id', StringType(), nullable=False),
        StructField('effective_date', DateType(), nullable=False),
        StructField('province', StringType(), nullable=False),
    ]
)

ORDERS_ROWS = [
    ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('125.00')),
    ('O002', 'C002', date(2026, 9, 1), 'CANCELLED', Decimal('80.00')),
    ('O003', 'C001', date(2026, 9, 2), 'COMPLETED', Decimal('45.00')),
    ('O004', 'C003', date(2026, 9, 2), 'COMPLETED', Decimal('200.00')),
    ('O005', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('30.00')),
]

CUSTOMERS_ROWS = [
    ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ('C002', 'Ben Tremblay', 'QC', 'CONSUMER'),
    ('C003', 'Carla Singh', 'BC', 'BUSINESS'),
    ('C004', 'Diego Martin', 'ON', 'CONSUMER'),
]

CUSTOMER_HISTORY_ROWS = [
    ('R001', 'C001', date(2026, 1, 1), 'ON'),
    ('R002', 'C001', date(2026, 6, 1), 'QC'),
    ('R003', 'C002', date(2026, 5, 1), 'AB'),
    ('R004', 'C002', date(2026, 5, 1), 'QC'),
]

orders_df = spark.createDataFrame(ORDERS_ROWS, schema=ORDERS_SCHEMA)
customers_df = spark.createDataFrame(CUSTOMERS_ROWS, schema=CUSTOMERS_SCHEMA)
customer_history_df = spark.createDataFrame(
    CUSTOMER_HISTORY_ROWS,
    schema=CUSTOMER_HISTORY_SCHEMA,
)

orders_df.orderBy('order_id').show(truncate=False)
customers_df.orderBy('customer_id').show(truncate=False)


<a id="architecture-exercise-protocol"></a>
# Architecture Exercise Protocol

For each experiment, ask:

```text
1. What responsibility is this code performing?
2. What inputs does it genuinely need?
3. What should it deliberately NOT know?
4. Does it perform side effects?
5. Can the business logic be called directly with in-memory DataFrames?
6. Is environment/run state explicit?
7. Is behavior deterministic?
8. Can a failure be localized?
9. Would another engineer know where to change this behavior?
```

The target dependency direction is approximately:

```text
configuration
      ↓
orchestration
  ├── readers ─────→ schemas
  ├── validation
  ├── transformations
  └── writers
```

Business transformations should stay near the center because they are the
easiest layer to reuse and test when infrastructure concerns do not leak into
them.

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Decompose a Monolithic Pipeline

A monolithic script often mixes:

```text
path selection
reading
schema assumptions
validation
business filters
joins
aggregation
logging
writing
error handling
```

The first architecture task is not to create seven files immediately. It is to
identify **responsibilities and boundaries**.

[Back to Table of Contents](#toc)


In [ ]:
MONOLITHIC_ANTI_PATTERN = (
    'def build_sales_report(spark):\n'
    '    environment = \'prod\'\n'
    '    orders_path = \'/prod/orders\'\n'
    '    customers_path = \'/prod/customers\'\n'
    '    output_path = \'/prod/sales_by_province\'\n'
    '\n'
    '    orders_df = spark.read.parquet(orders_path)\n'
    '    customers_df = spark.read.parquet(customers_path)\n'
    '\n'
    '    result_df = (\n'
    '        orders_df\n'
    '        .filter(F.col(\'order_status\') == \'COMPLETED\')\n'
    '        .join(customers_df, on=\'customer_id\', how=\'left\')\n'
    '        .groupBy(\'province\')\n'
    '        .agg(F.sum(\'net_sales\').alias(\'net_sales\'))\n'
    '    )\n'
    '\n'
    '    result_df.write.mode(\'overwrite\').parquet(output_path)\n'
)

print(MONOLITHIC_ANTI_PATTERN)

responsibility_map = {
    'environment and paths': 'configuration',
    'spark.read.parquet': 'reader / I/O',
    'order_status filter': 'business transformation',
    'customer join': 'business transformation',
    'province aggregation': 'business transformation',
    'write.parquet': 'writer / I/O',
}

for code_concern, responsibility in responsibility_map.items():
    print(f'{code_concern:28} -> {responsibility}')


**Worked conclusion**

The code works conceptually, but changing a path, output technology, business
rule, or test setup all requires editing the same function.

The refactoring target is:

```text
configuration → values
reader        → external data to DataFrame
validation    → correctness contracts
transformation→ DataFrame(s) to DataFrame
writer        → DataFrame to external persistence
orchestration → sequence those operations
```

No new abstraction is justified merely because a line of code exists.

[Back to Table of Contents](#toc)


<a id="experiment-2"></a>
# Experiment 2 — Reusable Transformation Functions

The stable center of the application should resemble:

```python
def transform_orders(orders_df, customers_df):
    ...
    return result_df
```

A transformation should depend on the data and explicit business parameters,
not on filesystem layout or runtime infrastructure.

[Back to Table of Contents](#toc)


In [ ]:
def require_columns(
    df: DataFrame,
    required_columns: set[str],
    label: str,
) -> None:
    '''Validate the structural columns required by one application boundary.'''

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f'{label} is missing required columns: {sorted(missing_columns)}'
        )


def transform_orders(
    orders_df: DataFrame,
    customers_df: DataFrame,
) -> DataFrame:
    '''Enrich order-grain rows with customer attributes.'''

    # WHAT: state the exact columns the transformation requires.
    # WHY: callers can reason about the contract without knowing file layout.
    require_columns(
        orders_df,
        {
            'order_id',
            'customer_id',
            'order_date',
            'order_status',
            'net_sales',
        },
        'orders_df',
    )
    require_columns(
        customers_df,
        {'customer_id', 'province', 'customer_segment'},
        'customers_df',
    )

    customer_projection_df = customers_df.select(
        'customer_id',
        'province',
        'customer_segment',
    )

    # Grain remains one row per order_id when customers are unique by customer_id.
    return orders_df.join(
        customer_projection_df,
        on='customer_id',
        how='left',
    )


enriched_orders_df = transform_orders(orders_df, customers_df)

enriched_orders_df.orderBy('order_id').show(truncate=False)


In [ ]:
# The transformation contract can be exercised directly with in-memory DataFrames.
assert enriched_orders_df.count() == orders_df.count()

expected_columns = {
    'order_id',
    'customer_id',
    'order_date',
    'order_status',
    'net_sales',
    'province',
    'customer_segment',
}

assert expected_columns.issubset(set(enriched_orders_df.columns))

print('Reusable transformation checks passed.')


**Worked conclusion**

`transform_orders()` knows:

```text
required columns
join semantics
business grain
```

It does not know:

```text
path
environment
credentials
write destination
scheduler
```

That is the desired separation.

[Back to Table of Contents](#toc)


<a id="experiment-3"></a>
# Experiment 3 — Parameterized Business Logic

A reusable function should accept legitimate run/business variation explicitly.

Bad hidden state:

```text
function silently reads a global status list
function checks machine environment variables mid-transformation
function calls today's date internally when replayability matters
```

Better:

```text
caller supplies the values as explicit parameters
```

[Back to Table of Contents](#toc)


In [ ]:
def filter_orders(
    orders_df: DataFrame,
    included_statuses: tuple[str, ...],
    minimum_net_sales: Decimal,
) -> DataFrame:
    '''Filter order-grain rows using explicit business parameters.'''

    require_columns(
        orders_df,
        {'order_status', 'net_sales'},
        'orders_df',
    )

    # Explicit parameters make the business behavior visible at the call site.
    return orders_df.filter(
        F.col('order_status').isin(*included_statuses)
        & (F.col('net_sales') >= F.lit(minimum_net_sales))
    )


completed_orders_df = filter_orders(
    orders_df,
    included_statuses=('COMPLETED',),
    minimum_net_sales=Decimal('0.00'),
)

high_value_completed_orders_df = filter_orders(
    orders_df,
    included_statuses=('COMPLETED',),
    minimum_net_sales=Decimal('100.00'),
)

print('All completed orders')
completed_orders_df.orderBy('order_id').show(truncate=False)

print('High-value completed orders')
high_value_completed_orders_df.orderBy('order_id').show(truncate=False)


In [ ]:
assert completed_orders_df.count() == 4
assert high_value_completed_orders_df.count() == 2

print('Parameterization checks passed.')


**Worked conclusion**

The function remains reusable because the changing values are explicit.

Do not make every constant configurable. Parameterize values that **legitimately
vary** across runs, environments, or supported business modes.

[Back to Table of Contents](#toc)


<a id="experiment-4"></a>
# Experiment 4 — Environment-Specific Configuration

Environment-specific infrastructure values should change without changing
transformation code.

We use an immutable configuration object so one pipeline execution has a clear
set of settings.

[Back to Table of Contents](#toc)


In [ ]:
class ConfigurationError(ValueError):
    '''Raised when application configuration is invalid.'''


@dataclass(frozen=True)
class PipelineConfig:
    '''Immutable configuration for one pipeline execution.'''

    environment: str
    orders_path: str
    customers_path: str
    output_path: str
    included_statuses: tuple[str, ...]
    minimum_net_sales: Decimal
    run_date: date
    write_mode: str = 'overwrite'


def build_config(
    environment: str,
    base_dir: Path,
) -> PipelineConfig:
    '''Build environment-specific settings without changing business logic.'''

    allowed_environments = {'dev', 'test', 'prod'}

    if environment not in allowed_environments:
        raise ConfigurationError(
            f'Unsupported environment {environment!r}. '
            f'Expected one of {sorted(allowed_environments)}.'
        )

    environment_root = base_dir / environment

    return PipelineConfig(
        environment=environment,
        orders_path=str(environment_root / 'input' / 'orders'),
        customers_path=str(environment_root / 'input' / 'customers'),
        output_path=str(environment_root / 'output' / 'sales_by_province'),
        included_statuses=('COMPLETED',),
        minimum_net_sales=Decimal('0.00'),
        # Inject a fixed run date so reruns and tests remain deterministic.
        run_date=date(2026, 9, 7),
    )


In [ ]:
with TemporaryDirectory() as temporary_directory:
    base_dir = Path(temporary_directory)

    for environment in ('dev', 'test', 'prod'):
        config = build_config(environment, base_dir)
        print(
            environment,
            config.orders_path,
            config.customers_path,
            config.output_path,
            config.run_date,
        )

# The business functions are unchanged for dev, test, and prod.
print(transform_orders.__name__)
print(filter_orders.__name__)


**Worked conclusion**

Environment configuration may legitimately own:

```text
paths
table names
run dates
write modes
environment name
supported configurable thresholds
```

It should not become a dumping ground for implementation details, and ordinary
configuration files should not contain committed secrets.

[Back to Table of Contents](#toc)


<a id="experiment-5"></a>
# Experiment 5 — Reader and Writer Boundaries

Readers translate external storage into DataFrames.

Writers translate DataFrames into external persistence.

Neither should redefine the business result.

[Back to Table of Contents](#toc)


In [ ]:
def read_orders(
    spark_session: SparkSession,
    path: str,
) -> DataFrame:
    '''Read orders using the declared application schema.'''

    return (
        spark_session.read
        .schema(ORDERS_SCHEMA)
        .parquet(path)
    )


def read_customers(
    spark_session: SparkSession,
    path: str,
) -> DataFrame:
    '''Read customers using the declared application schema.'''

    return (
        spark_session.read
        .schema(CUSTOMERS_SCHEMA)
        .parquet(path)
    )


def write_sales_by_province(
    result_df: DataFrame,
    path: str,
    mode: str,
) -> None:
    '''Persist the final province-grain result.'''

    (
        result_df.write
        .mode(mode)
        .parquet(path)
    )


In [ ]:
with TemporaryDirectory() as temporary_directory:
    base_dir = Path(temporary_directory)
    config = build_config('test', base_dir)

    # Seed deterministic source data only for the local teaching environment.
    orders_df.write.mode('overwrite').parquet(config.orders_path)
    customers_df.write.mode('overwrite').parquet(config.customers_path)

    reread_orders_df = read_orders(spark, config.orders_path)
    reread_customers_df = read_customers(spark, config.customers_path)

    assert reread_orders_df.schema == ORDERS_SCHEMA
    assert reread_customers_df.schema == CUSTOMERS_SCHEMA

    print('Reader boundary checks passed.')


**Worked conclusion**

A migration from Parquet to another source/sink should primarily affect the I/O
boundary.

The transformation layer should not need to be rewritten merely because the
storage technology changes.

[Back to Table of Contents](#toc)


<a id="experiment-6"></a>
# Experiment 6 — Thin Orchestration

Orchestration inside the Spark application should read like the pipeline
workflow.

It coordinates responsibilities instead of containing the detailed business
logic itself.

[Back to Table of Contents](#toc)


In [ ]:
def build_sales_by_province(
    enriched_orders_df: DataFrame,
) -> DataFrame:
    '''Aggregate enriched order-grain rows to one row per province.'''

    require_columns(
        enriched_orders_df,
        {'order_id', 'province', 'net_sales'},
        'enriched_orders_df',
    )

    # Grain intentionally changes from order_id to province.
    return (
        enriched_orders_df
        .groupBy('province')
        .agg(
            F.countDistinct('order_id').alias('order_count'),
            F.sum('net_sales').alias('net_sales'),
        )
    )


def add_processing_date(
    result_df: DataFrame,
    run_date: date,
) -> DataFrame:
    '''Add an explicitly injected processing date.'''

    return result_df.withColumn(
        'processing_date',
        F.lit(run_date).cast('date'),
    )


def build_business_result(
    orders_input_df: DataFrame,
    customers_input_df: DataFrame,
    config: PipelineConfig,
) -> DataFrame:
    '''Coordinate business transformations without performing I/O.'''

    filtered_orders_df = filter_orders(
        orders_input_df,
        included_statuses=config.included_statuses,
        minimum_net_sales=config.minimum_net_sales,
    )

    enriched_orders_df = transform_orders(
        filtered_orders_df,
        customers_input_df,
    )

    sales_by_province_df = build_sales_by_province(enriched_orders_df)

    return add_processing_date(
        sales_by_province_df,
        run_date=config.run_date,
    )


In [ ]:
with TemporaryDirectory() as temporary_directory:
    config = build_config('test', Path(temporary_directory))

    result_df = build_business_result(
        orders_df,
        customers_df,
        config,
    )

    result_df.orderBy('province').show(truncate=False)


**Worked conclusion**

The orchestration flow is easy to scan:

```text
filter
→ enrich
→ aggregate
→ attach run metadata
```

Detailed join/aggregation mechanics remain in reusable functions.

[Back to Table of Contents](#toc)


<a id="experiment-7"></a>
# Experiment 7 — Useful Logging

Logging should describe application state and major boundaries without creating
unnecessary Spark jobs solely for log decoration.

[Back to Table of Contents](#toc)


In [ ]:
def configure_logging() -> logging.Logger:
    '''Configure one application logger at the application boundary.'''

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    )

    return logging.getLogger('phase_09')


logger = configure_logging()

logger.info('Pipeline boundary example started')
logger.info('Environment selected | environment=%s', 'test')
logger.info('Business step selected | step=%s', 'sales_by_province')
logger.info('Pipeline boundary example completed')


Do **not** automatically do this merely for a log message:

```python
logger.info('Rows=%s', df.count())
```

`count()` is a real Spark action. Run it only when the count itself is an
intentional validation/metric requirement.

A useful log normally captures:

```text
pipeline start/finish
environment/run parameters
major processing boundary
input/output location
failure context
```

[Back to Table of Contents](#toc)


<a id="experiment-8"></a>
# Experiment 8 — Exception Handling

Good exception handling preserves failure information.

Catch only when you can:

```text
recover
add useful context
translate to a meaningful application-level error
```

Do not catch every exception just to print a message and continue.

[Back to Table of Contents](#toc)


In [ ]:
class PipelineExecutionError(RuntimeError):
    '''Raised when orchestration adds pipeline-level failure context.'''


def run_business_pipeline_safely(
    orders_input_df: DataFrame,
    customers_input_df: DataFrame,
    config: PipelineConfig,
) -> DataFrame:
    '''Add pipeline context while preserving the original failure as the cause.'''

    try:
        return build_business_result(
            orders_input_df,
            customers_input_df,
            config,
        )
    except ConfigurationError:
        # Already meaningful and specific; preserve the original type.
        raise
    except Exception as exc:
        raise PipelineExecutionError(
            f'Business pipeline failed in environment {config.environment!r}.'
        ) from exc


In [ ]:
broken_orders_df = orders_df.drop('order_status')

with TemporaryDirectory() as temporary_directory:
    config = build_config('test', Path(temporary_directory))

    try:
        run_business_pipeline_safely(
            broken_orders_df,
            customers_df,
            config,
        )
    except PipelineExecutionError as exc:
        print(type(exc).__name__)
        print(str(exc))
        print('Original cause:', type(exc.__cause__).__name__)
        print('Cause message:', str(exc.__cause__))


**Worked conclusion**

The caller receives:

```text
application context
+
original exception chain
```

That is much more maintainable than swallowing the error or replacing it with
an unrelated generic message.

[Back to Table of Contents](#toc)


<a id="experiment-9"></a>
# Experiment 9 — Deterministic Behavior

Determinism means:

```text
same logical inputs + same explicit parameters
→ same logical business output
```

This requires attention to tie-breaking, run-dependent values, randomness, and
implicit assumptions about row order.

[Back to Table of Contents](#toc)


In [ ]:
def select_latest_customer_record(
    history_df: DataFrame,
) -> DataFrame:
    '''Select one deterministic latest history row per customer.'''

    require_columns(
        history_df,
        {
            'customer_record_id',
            'customer_id',
            'effective_date',
            'province',
        },
        'history_df',
    )

    latest_window = (
        Window
        .partitionBy('customer_id')
        # customer_record_id provides a stable tie-breaker for equal dates.
        .orderBy(
            F.col('effective_date').desc(),
            F.col('customer_record_id').desc(),
        )
    )

    return (
        history_df
        .withColumn('row_number', F.row_number().over(latest_window))
        .filter(F.col('row_number') == 1)
        .drop('row_number')
    )


latest_customer_df = select_latest_customer_record(customer_history_df)

latest_customer_df.orderBy('customer_id').show(truncate=False)


In [ ]:
latest_rows = {
    row['customer_id']: row['customer_record_id']
    for row in latest_customer_df.collect()
}

assert latest_rows == {
    'C001': 'R002',
    'C002': 'R004',
}

# The run date is injected instead of calling current_date() inside business logic.
fixed_run_date = date(2026, 9, 7)
dated_df = add_processing_date(
    spark.createDataFrame([('ON',)], ['province']),
    fixed_run_date,
)

assert dated_df.first()['processing_date'] == fixed_run_date

print('Determinism checks passed.')


A display `orderBy()` is not the same thing as deterministic business logic.
The important issue is whether ambiguous business choices have stable rules.

Examples:

```text
latest record → explicit tie-breaker
run date      → injected parameter
randomness    → seed when intentional
business key  → stable deterministic key
```

[Back to Table of Contents](#toc)


<a id="experiment-10"></a>
# Experiment 10 — Testable Transformation Design

A well-designed transformation can be tested with tiny in-memory DataFrames.

The test does not need:

```text
production paths
credentials
cloud services
scheduler
real output table
```

[Back to Table of Contents](#toc)


In [ ]:
with TemporaryDirectory() as temporary_directory:
    config = build_config('test', Path(temporary_directory))

    actual_df = build_business_result(
        orders_df,
        customers_df,
        config,
    )

    actual_rows = [
        (
            row['province'],
            row['order_count'],
            row['net_sales'],
            row['processing_date'],
        )
        for row in actual_df.orderBy('province').collect()
    ]

expected_rows = [
    ('BC', 1, Decimal('200.00'), date(2026, 9, 7)),
    ('ON', 3, Decimal('200.00'), date(2026, 9, 7)),
]

assert actual_rows == expected_rows

# Grain check: exactly one row per province.
assert actual_df.groupBy('province').count().filter(F.col('count') > 1).count() == 0

print('Transformation test seam works without external I/O.')


**Worked conclusion**

Phase 11 will formalize these checks with `pytest`, fixtures, and reusable
DataFrame comparison strategies.

Phase 9's responsibility is to create the **test seam** now.

[Back to Table of Contents](#toc)


<a id="experiment-11"></a>
# Experiment 11 — Dependency Direction and Coupling

A maintainable dependency graph should keep business logic from depending
outward on infrastructure.

Preferred direction:

```text
pipeline/orchestration
  ├── config
  ├── readers ─────→ schemas
  ├── validation
  ├── transformations
  └── writers
```

Transformations should not import the pipeline entry point, inspect deployment
environment state, or decide storage destinations.

[Back to Table of Contents](#toc)


In [ ]:
architecture_contracts = {
    'config': {
        'knows': 'environment/run values',
        'must_not_know': 'DataFrame transformation implementation',
    },
    'reader': {
        'knows': 'format, schema, source location',
        'must_not_know': 'business aggregation rules',
    },
    'transformation': {
        'knows': 'DataFrame columns, business rules, grain',
        'must_not_know': 'paths, credentials, scheduler, write destination',
    },
    'writer': {
        'knows': 'persistence mechanics and destination',
        'must_not_know': 'how business measures are calculated',
    },
    'pipeline': {
        'knows': 'application sequence',
        'must_not_know': 'every join/filter implementation detail',
    },
}

for layer, contract in architecture_contracts.items():
    knows = contract['knows']
    avoids = contract['must_not_know']
    print(f'{layer:15} knows -> {knows}')
    print(f'               avoids -> {avoids}')


**Worked conclusion**

Coupling is not automatically bad—software components must collaborate.

The goal is to avoid **unnecessary knowledge** crossing boundaries.

A useful question is:

> If the output moves from Parquet to a warehouse, which layer should change?

Primarily the writer/configuration boundary. The completed-order filter and
province aggregation should not need rewriting.

[Back to Table of Contents](#toc)


<a id="experiment-12"></a>
# Experiment 12 — Dependency Management and Packaging

Dependency management answers:

```text
What external software does this application require?
```

Packaging answers:

```text
How is the application code organized as an importable/deployable unit?
```

These concerns are related but not identical.

[Back to Table of Contents](#toc)


In [ ]:
dependency_reference = [
    'pyspark==4.2.0',
    'ipykernel',
]

print('requirements.txt reference')
for dependency in dependency_reference:
    print(dependency)

package_layout = (
    'src/\n'
    '└── pyspark_learning/\n'
    '    └── phase_09_app/\n'
    '        ├── __init__.py\n'
    '        ├── config.py\n'
    '        ├── schemas.py\n'
    '        ├── readers.py\n'
    '        ├── validation.py\n'
    '        ├── transformations.py\n'
    '        ├── writers.py\n'
    '        └── pipeline.py\n'
    '\n'
    'tests/\n'
    '├── test_validation.py\n'
    '└── test_transformations.py\n'
)

print(package_layout)


The notebook does **not** create that package yet.

The structure is justified when the implementation is worth preserving as an
application, not merely because a curriculum diagram contains those filenames.

A package should make imports and deployment clearer. It should not create
layers whose only purpose is forwarding one function call to another.

[Back to Table of Contents](#toc)


<a id="applied-project"></a>
# Applied Phase 9 Project

## Refactor a Retail Pipeline Into an Application

This integrated run combines the boundaries from the previous experiments.

Target workflow:

```text
configuration
    ↓
read orders/customers
    ↓
validate required keys
    ↓
filter orders
    ↓
enrich orders
    ↓
aggregate to province
    ↓
add deterministic run date
    ↓
write output
```

Expected output grain:

```text
one row per province
```

The orchestration function should be understandable without reading the
implementation of every transformation.

[Back to Table of Contents](#toc)


In [ ]:
def require_non_null_keys(
    df: DataFrame,
    key_columns: list[str],
    label: str,
) -> None:
    '''Fail when required business keys contain NULL values.'''

    null_condition = F.lit(False)

    for column_name in key_columns:
        null_condition = null_condition | F.col(column_name).isNull()

    # This action is intentional because row-level key validity is being checked.
    invalid_exists = df.filter(null_condition).limit(1).count() > 0

    if invalid_exists:
        raise ValueError(
            f'{label} contains NULL values in required key columns {key_columns}.'
        )


def run_pipeline(
    spark_session: SparkSession,
    config: PipelineConfig,
) -> DataFrame:
    '''Coordinate I/O and business layers for one application execution.'''

    logger.info(
        'Pipeline started | environment=%s | run_date=%s',
        config.environment,
        config.run_date,
    )

    try:
        logger.info('Reading orders | path=%s', config.orders_path)
        input_orders_df = read_orders(
            spark_session,
            config.orders_path,
        )

        logger.info('Reading customers | path=%s', config.customers_path)
        input_customers_df = read_customers(
            spark_session,
            config.customers_path,
        )

        require_non_null_keys(
            input_orders_df,
            ['order_id', 'customer_id'],
            'orders_df',
        )
        require_non_null_keys(
            input_customers_df,
            ['customer_id'],
            'customers_df',
        )

        final_df = build_business_result(
            input_orders_df,
            input_customers_df,
            config,
        )

        logger.info('Writing output | path=%s', config.output_path)
        write_sales_by_province(
            final_df,
            path=config.output_path,
            mode=config.write_mode,
        )

    except ConfigurationError:
        raise
    except Exception as exc:
        raise PipelineExecutionError(
            f'Phase 9 pipeline failed in environment {config.environment!r}.'
        ) from exc

    logger.info('Pipeline completed | environment=%s', config.environment)

    return final_df


In [ ]:
with TemporaryDirectory() as temporary_directory:
    base_dir = Path(temporary_directory)
    config = build_config('dev', base_dir)

    # Local seed setup is outside production business logic.
    orders_df.write.mode('overwrite').parquet(config.orders_path)
    customers_df.write.mode('overwrite').parquet(config.customers_path)

    final_df = run_pipeline(
        spark,
        config,
    )

    print('Returned DataFrame')
    final_df.orderBy('province').show(truncate=False)

    persisted_df = spark.read.parquet(config.output_path)

    print('Persisted output')
    persisted_df.orderBy('province').show(truncate=False)

    expected_rows = [
        ('BC', 1, Decimal('200.00'), date(2026, 9, 7)),
        ('ON', 3, Decimal('200.00'), date(2026, 9, 7)),
    ]

    actual_rows = [
        (
            row['province'],
            row['order_count'],
            row['net_sales'],
            row['processing_date'],
        )
        for row in persisted_df.orderBy('province').collect()
    ]

    assert actual_rows == expected_rows
    assert persisted_df.schema == final_df.schema

print('Applied Phase 9 pipeline checks passed.')


## Applied Architecture Review

The completed design now has clear answers:

```text
Where is the completed-order rule?
→ filter_orders()

Where are environment paths defined?
→ PipelineConfig / build_config()

Which functions are directly testable with in-memory DataFrames?
→ filter_orders()
→ transform_orders()
→ build_sales_by_province()
→ add_processing_date()
→ build_business_result()
→ select_latest_customer_record()

Which functions intentionally perform I/O?
→ read_orders()
→ read_customers()
→ write_sales_by_province()

Where is logging configured?
→ configure_logging()

Where is pipeline-level failure context added?
→ run_pipeline()

How is run_date deterministic?
→ supplied through PipelineConfig

What dependencies are declared externally?
→ requirements.txt

What changes if Parquet output becomes a warehouse write?
→ writer/configuration layer

What should remain unchanged?
→ business transformation logic
```

### Success standard

Someone unfamiliar with the original author should be able to identify:

```text
business rules
schemas
I/O boundaries
configuration
orchestration
side effects
test seams
failure boundaries
deterministic inputs
dependency/package expectations
```

without reverse-engineering a monolithic script.

This notebook is **practice evidence**, not the formal mastery gate.

[Back to Table of Contents](#toc)


<a id="cleanup"></a>
# Cleanup

Stop the local Spark application after completing the notebook.

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()

print('Phase 9 solution notebook complete.')
